# Error analysis & ablative analysis

_Machine Learning — more_

**Find which part of your system to fix — and which part actually earns its keep.**

Real systems are pipelines: several components in a row (preprocess → detect → classify …).

     When the whole thing underperforms, where do you spend your effort? Guessing wastes weeks.

---

This notebook is a practice scaffold for the **Error analysis & ablative analysis** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X, y = make_classification(n_samples=400, n_features=8, n_informative=4,
                           n_redundant=0, random_state=0)
acc = lambda cols: cross_val_score(
    LogisticRegression(max_iter=1000), X[:, cols], y, cv=5).mean()

full = acc(list(range(8)))                       # baseline: all features
print("full system accuracy = %.4f" % full)

blocks = {"feat0-1": [0, 1], "feat2-3": [2, 3],
          "feat4-5": [4, 5], "feat6-7": [6, 7]}
drops = {}
for name, cols in blocks.items():
    keep = [c for c in range(8) if c not in cols]
    drop = full - acc(keep)                       # accuracy lost without block
    drops[name] = drop
    print("remove %-8s -> drop = %+.4f" % (name, drop))
top = max(drops, key=drops.get)
print("most important block = %s (largest drop)" % top)


## Visualize the data & results

_Which block of chemical features actually earns its keep?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# REAL data: 178 wines, 13 chemical features, 3 cultivars.
wn = load_wine()
X = StandardScaler().fit_transform(wn.data)
y = wn.target
acc = lambda cols: cross_val_score(
    LogisticRegression(max_iter=5000), X[:, cols], y, cv=5).mean()

full = acc(list(range(13)))              # baseline: all 13 features
blocks = {'alcohol/acid': [0, 1, 2], 'ash/Mg': [3, 4, 5],
          'phenols': [6, 7, 8], 'color/hue': [9, 10, 11], 'proline': [12]}
names, drops = [], []
for name, cols in blocks.items():
    keep = [c for c in range(13) if c not in cols]
    names.append(name)
    drops.append(full - acc(keep))       # accuracy lost without that block

colors = ['#ffb454' if d == max(drops) else '#4ea1ff' for d in drops]
plt.bar(names, [d*100 for d in drops], color=colors)
plt.ylabel('accuracy lost (%)')
plt.title('Wine: accuracy lost when each chemical feature block is removed')
plt.show()
